# Análise Exploratória - Extração de Features no dataset original dos Lotes 1, 2 e 3

Autor:  Viviane da Rosa Sommer

Atualizado: 14/01/2025

## Objetivo:
Notebook para extrair de diferentes maneiras as características das imagens, para ver quais são mais interessantes para no futuro clusterizar nossas imagens.

## Importações Necessárias

In [ ]:
! pip install numpy tqdm pillow torch torchvision scikit-image scikit-learn umap-learn opencv-python
! export TRANSFORMERS_OFFLINE=1
import os
import numpy as np
from tqdm import tqdm
from PIL import Image
import cv2
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import SwinModel, AutoFeatureExtractor, CLIPProcessor, CLIPModel

## Declaração de Constantes e Modelos

In [ ]:
DATASET_PATH = "dataset"
IMAGE_PATH = os.path.join(DATASET_PATH, "images")
OUTPUT_FEATURES_PATH = "features" 
os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)

## Funções Necessárias

In [ ]:
def extract_histogram(image, bins=(8, 8, 8)):
    """
    Extracts the color histogram of the image in RGB.
    :param image: Input image (PIL.Image).
    :param bins: Tuple specifying the number of bins for each channel (default: (8, 8, 8)).
    :return: Flattened histogram as a NumPy array.
    """
    histogram = cv2.calcHist([np.array(image)], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    histogram = cv2.normalize(histogram, histogram).flatten()
    return histogram


def extract_hog(image):
    """
    Extracts HOG features from the image.
    :param image: Input image (PIL.Image).
    :return: HOG features as a NumPy array.
    """
    image_gray = image.convert("L")
    image_resized = image_gray.resize((800, 800))
    features = hog(
        np.array(image_resized),
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return features


def extract_embedding(image, model, preprocess):
    """
    Extracts embeddings using a pre-trained Deep Learning model.
    :param image: Input image (PIL.Image).
    :param model: Pre-trained model for feature extraction.
    :param preprocess: Preprocessing function for the model.
    :return: Embedding as a NumPy array.
    """
    input_tensor = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        features = model(input_tensor)
    return features.squeeze(0).numpy()


def extract_swin_features(image):

    local_model_dir = "models/swin-base-patch4-window7-224"
    feature_extractor = AutoFeatureExtractor.from_pretrained(local_model_dir)
    model = SwinModel.from_pretrained(local_model_dir)
    model.eval()

    inputs = feature_extractor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    pooled_embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0).numpy()
    return pooled_embedding


def extract_clip_features(image):

    local_model_dir = "models/clip-vit-base-patch16"
    processor = CLIPProcessor.from_pretrained(local_model_dir)
    model = CLIPModel.from_pretrained(local_model_dir)
    model.eval()

    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    
    normalized_embedding = outputs / outputs.norm(dim=-1, keepdim=True)
    return normalized_embedding.squeeze(0).numpy()


def process_images(split, method="embedding", model=None, preprocess=None):
    """
    Processes training or testing images and extracts features based on the specified method.
    :param split: Dataset split ("train" or "test").
    :param method: Feature extraction method ("histogram", "hog", "embedding", "vit", "swin", "deit", "clip").
    :param model: Pre-trained model (required for "embedding" method).
    :param preprocess: Preprocessing function (required for "embedding" method").
    """
    image_dir = os.path.join(IMAGE_PATH, split)
    output_dir = os.path.join(OUTPUT_FEATURES_PATH, split, method)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Processing images in folder: {image_dir} with method: {method}")
    for image_file in tqdm(os.listdir(image_dir)):
        if image_file.endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(image_dir, image_file)
            
            try:
                image = Image.open(image_path).convert("RGB")

                if method == "histogram":
                    features = extract_histogram(image)
                elif method == "hog":
                    features = extract_hog(image)
                elif method == "embedding":
                    if model is None or preprocess is None:
                        raise ValueError("Model and preprocessing must be provided for embeddings.")
                    features = extract_embedding(image, model, preprocess)
                elif method == "swin":
                    features = extract_swin_features(image)
                elif method == "clip":
                    features = extract_clip_features(image)
                else:
                    raise ValueError(f"Unknown extraction method: {method}")

                feature_file = os.path.splitext(image_file)[0] + ".npy"
                np.save(os.path.join(output_dir, feature_file), features)

            except Exception as e:
                print(f"Error processing {image_file}: {e}")


def reduce_dimensionality(features, method="pca", n_components=2):
    """
    Reduces the dimensionality of features using PCA, t-SNE, or UMAP.
    :param features: Input features as a NumPy array.
    :param method: Dimensionality reduction method ("pca", "tsne", or "umap").
    :param n_components: Number of components to reduce to.
    :return: Reduced features as a NumPy array.
    """
    if method == "pca":
        reducer = PCA(n_components=n_components)
    elif method == "tsne":
        reducer = TSNE(n_components=n_components, random_state=42)
    elif method == "umap":
        reducer = umap.UMAP(n_components=n_components, random_state=42)
    else:
        raise ValueError(f"Unknown dimensionality reduction method: {method}")

    print(f"Reducing dimensionality using {method.upper()} to {n_components} components...")
    reduced_features = reducer.fit_transform(features)
    return reduced_features


def plot_reduced_features(features, labels=None, title="Dimensionality Reduction", plot_3d=False):
    """
    Plots reduced features in 2D or 3D.
    :param features: 2D array with reduced features (n_samples, 2 or 3).
    :param labels: (Optional) List or array with labels for coloring the points.
    :param title: Plot title.
    :param plot_3d: If True, creates a 3D plot. Otherwise, creates a 2D plot.
    """
    if plot_3d and features.shape[1] != 3:
        raise ValueError("To plot in 3D, features must have 3 dimensions.")
    
    plt.figure(figsize=(10, 8))
    
    if plot_3d:
        ax = plt.axes(projection='3d')
        if labels is not None:
            scatter = ax.scatter(
                features[:, 0], features[:, 1], features[:, 2],
                c=labels, cmap="tab10", s=50, alpha=0.7
            )
            plt.colorbar(scatter, ax=ax, label="Clusters/Classes")
        else:
            ax.scatter(features[:, 0], features[:, 1], features[:, 2], s=50, alpha=0.7)
        
        ax.set_xlabel("Component 1", fontsize=12)
        ax.set_ylabel("Component 2", fontsize=12)
        ax.set_zlabel("Component 3", fontsize=12)
        ax.set_title(title, fontsize=16)
        plt.show()
    else:
        if labels is not None:
            sns.scatterplot(
                x=features[:, 0], 
                y=features[:, 1], 
                hue=labels, 
                palette="tab10", 
                s=50, 
                alpha=0.7
            )
            plt.legend(title="Clusters/Classes", bbox_to_anchor=(1.05, 1), loc='upper left')
        else:
            plt.scatter(features[:, 0], features[:, 1], s=50, alpha=0.7)
        
        plt.title(title, fontsize=16)
        plt.xlabel("Component 1", fontsize=14)
        plt.ylabel("Component 2", fontsize=14)
        plt.grid(alpha=0.3)
        plt.show()


def process_dimensionality_reduction_and_plot(method="pca", n_components=2):
    """
    Applies dimensionality reduction and plots the reduced data from the combined dataset.
    :param method: Dimensionality reduction method ("pca", "tsne", or "umap").
    :param n_components: Number of components to reduce to.
    """
    reduced_dir = os.path.join(OUTPUT_FEATURES_PATH, "reduced", method)
    os.makedirs(reduced_dir, exist_ok=True)

    for feature_method in ["histogram", "hog", "embedding", "swin", "clip"]:
        feature_dir = os.path.join(OUTPUT_FEATURES_PATH, "combined", feature_method)
        if not os.path.exists(feature_dir):
            print(f"Feature directory not found: {feature_dir}")
            continue

        print(f"Loading features from {feature_dir}...")
        features = []
        files = []

        for feature_file in os.listdir(feature_dir):
            if feature_file.endswith(".npy"):
                file_path = os.path.join(feature_dir, feature_file)
                features.append(np.load(file_path))
                files.append(feature_file)

        if not features:
            print(f"No features found in {feature_dir}.")
            continue

        features = np.array(features)
        reduced_features = reduce_dimensionality(features, method=method, n_components=n_components)

        output_dir = os.path.join(reduced_dir, "combined", feature_method)
        os.makedirs(output_dir, exist_ok=True)
        for file, reduced in zip(files, reduced_features):
            output_file = os.path.join(output_dir, file)
            np.save(output_file, reduced)

        print(f"Plotting results in 2D for {feature_method} with {method.upper()}")
        plot_reduced_features(
            reduced_features,
            labels=None,
            title=f"{method.upper()} - {feature_method} (combined) - 2D",
            plot_3d=False
        )

        if n_components >= 3:
            print(f"Plotting results in 3D for {feature_method} with {method.upper()}")
            plot_reduced_features(
                reduced_features,
                labels=None,
                title=f"{method.upper()} - {feature_method} (combined) - 3D",
                plot_3d=True
            )



def load_resnet_model(weights_path):
    """
    Loads a pre-trained ResNet50 model with the specified weights.
    :param weights_path: Path to the pre-trained weights file.
    :return: Pre-trained ResNet50 model with the fully connected layer removed.
    """
    model = models.resnet50()
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    model.fc = torch.nn.Identity()  
    model.eval()  
    return model


def initialize_preprocess():
    """
    Initializes the preprocessing pipeline for the ResNet model.
    :return: Preprocessing pipeline as a torchvision.transforms.Compose object.
    """
    return transforms.Compose([
        transforms.Resize((800, 800)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def process_images_combined(method="embedding", model=None, preprocess=None):
    """
    Processes images from both 'train' and 'test' folders as a single combined dataset and extracts features.
    :param method: Feature extraction method ("histogram", "hog", "embedding", "vit", "swin", "deit", "clip").
    :param model: Pre-trained model (required for "embedding" method).
    :param preprocess: Preprocessing pipeline (required for "embedding" method).
    """
    train_dir = os.path.join(IMAGE_PATH, "train")
    test_dir = os.path.join(IMAGE_PATH, "test")
    combined_dir = [train_dir, test_dir]  

    output_dir = os.path.join(OUTPUT_FEATURES_PATH, "combined", method)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Processing images from 'train' and 'test' as a combined dataset with method: {method}")

    for folder in combined_dir:
        for image_file in tqdm(os.listdir(folder)):
            if image_file.endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(folder, image_file)
                
                try:
                    image = Image.open(image_path).convert("RGB")

                    # Adicionando os novos métodos
                    if method == "histogram":
                        features = extract_histogram(image)
                    elif method == "hog":
                        features = extract_hog(image)
                    elif method == "embedding":
                        if model is None or preprocess is None:
                            raise ValueError("Model and preprocess must be provided for embeddings.")
                        features = extract_embedding(image, model, preprocess)
                    elif method == "swin":
                        features = extract_swin_features(image)
                    elif method == "clip":
                        features = extract_clip_features(image)
                    else:
                        raise ValueError(f"Unknown extraction method: {method}")

                    feature_file = os.path.splitext(image_file)[0] + ".npy"
                    np.save(os.path.join(output_dir, feature_file), features)

                except Exception as e:
                    print(f"Error processing {image_file} from folder {folder}: {e}")
                

def apply_dimensionality_reduction_and_plot(reduction_methods, n_components=3):
    """
    Applies dimensionality reduction and plots the results for all methods and splits.
    :param reduction_methods: List of dimensionality reduction methods ("pca", "tsne", "umap").
    :param n_components: Number of components to reduce to.
    """
    for reduction_method in reduction_methods:
        process_dimensionality_reduction_and_plot(method=reduction_method, n_components=n_components)

## Extração das características, e análise dos resultados

In [ ]:
weights_path = "models/resnet50-0676ba61.pth"
resnet_model = load_resnet_model(weights_path)
preprocess = initialize_preprocess()

feature_extraction_methods =  ["histogram", "hog", "embedding", "swin", "clip"]
dimensionality_reduction_methods = ["pca", "tsne", "umap"]

for method in feature_extraction_methods:
    process_images_combined(method=method, model=resnet_model, preprocess=preprocess if method == "embedding" else None)

apply_dimensionality_reduction_and_plot(dimensionality_reduction_methods, n_components=3)

print("Combined pipeline completed!")

In [ ]:
!jupyter nbconvert --to html Analise_Extração_Features.ipynb